# 📊 Regresión: EDA + RandomForest, XGBoost, MLP, Keras y PyTorch

**Objetivo educativo:** aprender el flujo completo de un problema de regresión.

En este notebook trabajaremos con el dataset **California Housing** (precio medio de viviendas por distrito). Cubriremos:

1. **EDA** — análisis exploratorio y visualización.
2. **Detección y manejo de outliers** — método IQR y visualización con boxplots.
3. **Preprocesamiento** — escalado y división train/test.
4. **Modelos** — Random Forest, XGBoost, MLP (sklearn), Keras y PyTorch.
5. **Curvas de entrenamiento y prueba** — para diagnosticar sobreajuste/subajuste.
6. **Comparación** — de complejidad, tiempo y métricas (MSE, MAE, R²).

> 💡 Nota: la métrica **MAD** (Mean Absolute Deviation) es equivalente al **MAE** (Mean Absolute Error) en este contexto.


## 1. Importación de librerías

In [ ]:
# Librerías generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# XGBoost
from xgboost import XGBRegressor

# Keras / TensorFlow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 90
print("✅ Librerías cargadas correctamente")

## 2. Carga del dataset

Usaremos **California Housing**: características de distritos californianos y su precio medio de vivienda (en cientos de miles de USD).

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame
df.rename(columns={'MedHouseVal': 'Precio'}, inplace=True)
print(f"Forma del dataset: {df.shape}")
df.head()

In [ ]:
print("Descripción de las variables:")
print(data.DESCR[:1500])

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Información general
print("Tipos de datos y valores nulos:")
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe().T

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['Precio'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribución del Precio')
sns.boxplot(x=df['Precio'], ax=axes[1], color='salmon')
axes[1].set_title('Boxplot del Precio')
plt.tight_layout()
plt.show()

In [ ]:
# Histogramas de todas las variables
df.hist(bins=30, figsize=(14, 10), color='steelblue', edgecolor='black')
plt.suptitle('Distribuciones de las variables', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación
plt.figure(figsize=(10, 7))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', center=0, square=True)
plt.title('Matriz de correlación')
plt.show()

print("\nCorrelación con el Precio (ordenada):")
print(corr['Precio'].sort_values(ascending=False))

### 📌 Observaciones del EDA
- La variable `MedInc` (ingreso medio) es la más correlacionada positivamente con el precio.
- El precio parece **saturado** en 5.0 (posible tope superior) — un artefacto del dataset.
- Varias variables presentan sesgo (skew) y posibles outliers.


## 4. Detección y manejo de outliers

Usaremos el método del **rango intercuartílico (IQR)**: se consideran outliers los valores que caen fuera de `[Q1 - 1.5·IQR, Q3 + 1.5·IQR]`.

In [ ]:
def detectar_outliers_iqr(df, columnas):
    resumen = {}
    for col in columnas:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lim_inf = Q1 - 1.5 * IQR
        lim_sup = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lim_inf) | (df[col] > lim_sup)]
        resumen[col] = {
            'n_outliers': len(outliers),
            'porcentaje': 100 * len(outliers) / len(df),
            'lim_inf': lim_inf,
            'lim_sup': lim_sup
        }
    return pd.DataFrame(resumen).T

columnas_num = df.columns.drop('Precio')
outliers_df = detectar_outliers_iqr(df, columnas_num)
outliers_df.sort_values('n_outliers', ascending=False)

In [ ]:
# Boxplots antes de tratar outliers
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.ravel(), columnas_num):
    sns.boxplot(x=df[col], ax=ax, color='salmon')
    ax.set_title(col)
plt.suptitle('Boxplots ANTES del manejo de outliers', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Estrategia: winsorización (recorte a los límites IQR).
# Esto es preferible a eliminar filas cuando muchas columnas tienen outliers.
df_limpio = df.copy()
for col in columnas_num:
    Q1 = df_limpio[col].quantile(0.25)
    Q3 = df_limpio[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    df_limpio[col] = df_limpio[col].clip(lower=lim_inf, upper=lim_sup)

print(f"Filas originales: {len(df)}  |  Filas tras winsorización: {len(df_limpio)}")

# Boxplots después
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.ravel(), columnas_num):
    sns.boxplot(x=df_limpio[col], ax=ax, color='lightgreen')
    ax.set_title(col)
plt.suptitle('Boxplots DESPUÉS del manejo de outliers (winsorización)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### 💡 Nota sobre estrategias frente a outliers
| Estrategia | Cuándo usarla |
|---|---|
| **Eliminar filas** | Pocos outliers y datos abundantes |
| **Winsorizar (recortar)** | Muchos outliers en varias columnas |
| **Transformar** (log, sqrt) | Distribuciones muy sesgadas |
| **Mantener** | Los outliers son señal, no ruido |


## 5. Preprocesamiento: escalado y división train/test

In [ ]:
X = df_limpio.drop(columns='Precio').values
y = df_limpio['Precio'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# Escalado (importante para MLP, Keras y PyTorch)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 6. Función de evaluación

Calculamos **MSE**, **MAE (MAD)** y **R²** tanto en train como en test para diagnosticar sobreajuste.

In [ ]:
def evaluar(nombre, y_train_true, y_train_pred, y_test_true, y_test_pred, t_entreno=None):
    resultados = {
        'Modelo': nombre,
        'MSE_train': mean_squared_error(y_train_true, y_train_pred),
        'MSE_test':  mean_squared_error(y_test_true, y_test_pred),
        'MAE_train': mean_absolute_error(y_train_true, y_train_pred),
        'MAE_test':  mean_absolute_error(y_test_true, y_test_pred),
        'R2_train':  r2_score(y_train_true, y_train_pred),
        'R2_test':   r2_score(y_test_true, y_test_pred),
    }
    if t_entreno is not None:
        resultados['Tiempo (s)'] = round(t_entreno, 2)
    return resultados

resultados_globales = []
historias = {}  # para guardar curvas de aprendizaje

## 7. Modelo 1 · Random Forest

In [ ]:
t0 = time.time()
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)  # RF no necesita escalado
t_rf = time.time() - t0

y_tr_pred = rf.predict(X_train)
y_te_pred = rf.predict(X_test)
resultados_globales.append(evaluar('Random Forest', y_train, y_tr_pred, y_test, y_te_pred, t_rf))
print(f"⏱ Entrenamiento: {t_rf:.2f}s  |  R² test: {r2_score(y_test, y_te_pred):.3f}")

In [ ]:
# Curva de aprendizaje (train vs validación) para diagnóstico
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    X_train, y_train,
    cv=3, scoring='neg_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=-1
)
train_mse = -train_scores.mean(axis=1)
val_mse = -val_scores.mean(axis=1)
historias['Random Forest'] = {'sizes': train_sizes, 'train': train_mse, 'val': val_mse}

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mse, 'o-', label='Train MSE', color='steelblue')
plt.plot(train_sizes, val_mse,   'o-', label='Validación MSE', color='salmon')
plt.xlabel('Tamaño de entrenamiento'); plt.ylabel('MSE')
plt.title('Curva de aprendizaje — Random Forest')
plt.legend(); plt.grid(True); plt.show()

In [ ]:
# Importancia de features
importancias = pd.DataFrame({
    'feature': data.feature_names,
    'importancia': rf.feature_importances_
}).sort_values('importancia', ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(importancias['feature'], importancias['importancia'], color='steelblue')
plt.title('Importancia de variables — Random Forest')
plt.xlabel('Importancia'); plt.tight_layout(); plt.show()

## 8. Modelo 2 · XGBoost

Guardaremos las curvas por iteración con `eval_set` para diagnosticar sobreajuste directamente.

In [ ]:
t0 = time.time()
xgb = XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    random_state=SEED, n_jobs=-1,
    eval_metric='rmse'
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)
t_xgb = time.time() - t0

y_tr_pred = xgb.predict(X_train)
y_te_pred = xgb.predict(X_test)
resultados_globales.append(evaluar('XGBoost', y_train, y_tr_pred, y_test, y_te_pred, t_xgb))
print(f"⏱ Entrenamiento: {t_xgb:.2f}s  |  R² test: {r2_score(y_test, y_te_pred):.3f}")

In [ ]:
# Curvas de entrenamiento por iteración
evals_result = xgb.evals_result()
epochs = range(len(evals_result['validation_0']['rmse']))

plt.figure(figsize=(8, 5))
plt.plot(epochs, evals_result['validation_0']['rmse'], label='Train RMSE', color='steelblue')
plt.plot(epochs, evals_result['validation_1']['rmse'], label='Test RMSE', color='salmon')
plt.xlabel('Iteraciones (árboles)'); plt.ylabel('RMSE')
plt.title('Curvas de entrenamiento — XGBoost')
plt.legend(); plt.grid(True); plt.show()

historias['XGBoost'] = {
    'sizes': list(epochs),
    'train': evals_result['validation_0']['rmse'],
    'val':   evals_result['validation_1']['rmse']
}

## 9. Modelo 3 · MLP (Perceptrón multicapa · sklearn)

In [ ]:
t0 = time.time()
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=200,
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.1,
    verbose=False
)
mlp.fit(X_train_sc, y_train)
t_mlp = time.time() - t0

y_tr_pred = mlp.predict(X_train_sc)
y_te_pred = mlp.predict(X_test_sc)
resultados_globales.append(evaluar('MLP (sklearn)', y_train, y_tr_pred, y_test, y_te_pred, t_mlp))
print(f"⏱ Entrenamiento: {t_mlp:.2f}s  |  R² test: {r2_score(y_test, y_te_pred):.3f}")

# Curva de pérdida
plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, label='Pérdida entrenamiento', color='steelblue')
if hasattr(mlp, 'validation_scores_') and mlp.validation_scores_:
    plt.plot(mlp.validation_scores_, label='Score validación', color='salmon')
plt.xlabel('Iteración'); plt.ylabel('Pérdida / Score')
plt.title('Curva de entrenamiento — MLP sklearn')
plt.legend(); plt.grid(True); plt.show()

## 10. Modelo 4 · Red neuronal con Keras (TensorFlow)

In [ ]:
def crear_modelo_keras(n_features):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(n_features,)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1)  # salida lineal para regresión
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model_keras = crear_modelo_keras(X_train_sc.shape[1])
model_keras.summary()

In [ ]:
t0 = time.time()
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
hist_keras = model_keras.fit(
    X_train_sc, y_train,
    validation_split=0.2,
    epochs=100, batch_size=64,
    callbacks=[early_stop],
    verbose=0
)
t_keras = time.time() - t0

y_tr_pred = model_keras.predict(X_train_sc, verbose=0).ravel()
y_te_pred = model_keras.predict(X_test_sc,  verbose=0).ravel()
resultados_globales.append(evaluar('Keras NN', y_train, y_tr_pred, y_test, y_te_pred, t_keras))
print(f"⏱ Entrenamiento: {t_keras:.2f}s  |  Épocas: {len(hist_keras.history['loss'])}  |  R² test: {r2_score(y_test, y_te_pred):.3f}")

In [ ]:
# Curvas de entrenamiento Keras
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_keras.history['loss'], label='Train', color='steelblue')
axes[0].plot(hist_keras.history['val_loss'], label='Validación', color='salmon')
axes[0].set_title('Pérdida (MSE) — Keras'); axes[0].set_xlabel('Época'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(hist_keras.history['mae'], label='Train', color='steelblue')
axes[1].plot(hist_keras.history['val_mae'], label='Validación', color='salmon')
axes[1].set_title('MAE — Keras'); axes[1].set_xlabel('Época'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

historias['Keras NN'] = {
    'sizes': list(range(len(hist_keras.history['loss']))),
    'train': hist_keras.history['loss'],
    'val': hist_keras.history['val_loss']
}

## 11. Modelo 5 · Red neuronal con PyTorch

Implementamos manualmente el bucle de entrenamiento para que los estudiantes entiendan las piezas: `forward`, `loss`, `backward`, `step`.

In [ ]:
# División train/val dentro del train para curvas de aprendizaje
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_sc, y_train, test_size=0.2, random_state=SEED
)

# Tensores
X_tr_t  = torch.tensor(X_tr,  dtype=torch.float32)
y_tr_t  = torch.tensor(y_tr,  dtype=torch.float32).view(-1, 1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
X_te_t  = torch.tensor(X_test_sc, dtype=torch.float32)
y_te_t  = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)

# Definición del modelo
class MLPRegressorTorch(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

model_torch = MLPRegressorTorch(X_tr_t.shape[1])
optim  = torch.optim.Adam(model_torch.parameters(), lr=1e-3)
criter = nn.MSELoss()
print(model_torch)

In [ ]:
t0 = time.time()
n_epocas = 80
hist_torch = {'train_loss': [], 'val_loss': []}

for epoca in range(n_epocas):
    model_torch.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        optim.zero_grad()
        pred = model_torch(xb)
        loss = criter(pred, yb)
        loss.backward()
        optim.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= len(train_loader.dataset)
    
    model_torch.eval()
    with torch.no_grad():
        val_loss = criter(model_torch(X_val_t), y_val_t).item()
    
    hist_torch['train_loss'].append(epoch_loss)
    hist_torch['val_loss'].append(val_loss)
    
    if (epoca + 1) % 10 == 0:
        print(f"Época {epoca+1:3d} | Train MSE: {epoch_loss:.4f} | Val MSE: {val_loss:.4f}")

t_torch = time.time() - t0

# Predicciones
model_torch.eval()
with torch.no_grad():
    y_tr_pred_all = model_torch(torch.tensor(X_train_sc, dtype=torch.float32)).numpy().ravel()
    y_te_pred     = model_torch(X_te_t).numpy().ravel()

resultados_globales.append(evaluar('PyTorch NN', y_train, y_tr_pred_all, y_test, y_te_pred, t_torch))
print(f"\n⏱ Entrenamiento total: {t_torch:.2f}s  |  R² test: {r2_score(y_test, y_te_pred):.3f}")

In [ ]:
# Curvas de entrenamiento PyTorch
plt.figure(figsize=(8, 5))
plt.plot(hist_torch['train_loss'], label='Train', color='steelblue')
plt.plot(hist_torch['val_loss'], label='Validación', color='salmon')
plt.xlabel('Época'); plt.ylabel('MSE')
plt.title('Curvas de entrenamiento — PyTorch')
plt.legend(); plt.grid(True); plt.show()

historias['PyTorch NN'] = {
    'sizes': list(range(len(hist_torch['train_loss']))),
    'train': hist_torch['train_loss'],
    'val':   hist_torch['val_loss']
}

## 12. Comparativa final de los modelos

In [ ]:
df_resultados = pd.DataFrame(resultados_globales).set_index('Modelo')
df_resultados = df_resultados[['MSE_train','MSE_test','MAE_train','MAE_test','R2_train','R2_test','Tiempo (s)']]
df_resultados.round(4)

In [ ]:
# Diagnóstico de sobreajuste/subajuste
df_resultados['Gap_R2 (train - test)'] = df_resultados['R2_train'] - df_resultados['R2_test']

def diagnostico(row):
    if row['R2_train'] < 0.5 and row['R2_test'] < 0.5:
        return '⚠️ Subajuste'
    elif row['Gap_R2 (train - test)'] > 0.15:
        return '⚠️ Sobreajuste'
    else:
        return '✅ Buen balance'

df_resultados['Diagnóstico'] = df_resultados.apply(diagnostico, axis=1)
df_resultados[['R2_train','R2_test','Gap_R2 (train - test)','Diagnóstico']].round(3)

In [ ]:
# Gráfico comparativo de métricas
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

x = np.arange(len(df_resultados))
w = 0.35

axes[0].bar(x - w/2, df_resultados['MSE_train'], w, label='Train', color='steelblue')
axes[0].bar(x + w/2, df_resultados['MSE_test'],  w, label='Test',  color='salmon')
axes[0].set_xticks(x); axes[0].set_xticklabels(df_resultados.index, rotation=30)
axes[0].set_title('MSE'); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x - w/2, df_resultados['MAE_train'], w, label='Train', color='steelblue')
axes[1].bar(x + w/2, df_resultados['MAE_test'],  w, label='Test',  color='salmon')
axes[1].set_xticks(x); axes[1].set_xticklabels(df_resultados.index, rotation=30)
axes[1].set_title('MAE (MAD)'); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(x - w/2, df_resultados['R2_train'], w, label='Train', color='steelblue')
axes[2].bar(x + w/2, df_resultados['R2_test'],  w, label='Test',  color='salmon')
axes[2].set_xticks(x); axes[2].set_xticklabels(df_resultados.index, rotation=30)
axes[2].set_title('R²'); axes[2].legend(); axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Curvas de aprendizaje juntas (normalizando en X para poder compararlas)
plt.figure(figsize=(11, 6))
for nombre, h in historias.items():
    x = np.linspace(0, 1, len(h['train']))
    plt.plot(x, h['train'], '--', alpha=0.5, label=f'{nombre} — Train')
    plt.plot(x, h['val'],   '-',  label=f'{nombre} — Val/Test')
plt.xlabel('Progreso del entrenamiento (normalizado)')
plt.ylabel('Error (MSE / RMSE)')
plt.title('Comparativa de curvas de entrenamiento')
plt.legend(fontsize=8, loc='best'); plt.grid(True)
plt.yscale('log')  # escala log para visualizar mejor
plt.tight_layout(); plt.show()

## 13. Comparación de complejidad de los modelos

In [ ]:
# Estimación grosera del número de parámetros por modelo
params = {
    'Random Forest': sum(t.tree_.node_count for t in rf.estimators_),  # nodos totales
    'XGBoost':       xgb.get_booster().trees_to_dataframe().shape[0],  # nodos totales
    'MLP (sklearn)': sum(c.size for c in mlp.coefs_) + sum(b.size for b in mlp.intercepts_),
    'Keras NN':      model_keras.count_params(),
    'PyTorch NN':    sum(p.numel() for p in model_torch.parameters())
}

df_complejidad = pd.DataFrame({
    'Parámetros / Nodos': params,
    'Tiempo entreno (s)': df_resultados['Tiempo (s)'],
    'R² test': df_resultados['R2_test'].round(3)
})
df_complejidad

In [ ]:
# Gráfico: complejidad vs desempeño
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

x = np.arange(len(df_complejidad))
ax1.bar(x, df_complejidad['Parámetros / Nodos'], color='steelblue', alpha=0.6, label='Parámetros / Nodos')
ax1.set_yscale('log')
ax1.set_ylabel('Parámetros / Nodos (log)', color='steelblue')
ax1.set_xticks(x); ax1.set_xticklabels(df_complejidad.index, rotation=30)

ax2.plot(x, df_complejidad['R² test'], 'o-', color='crimson', linewidth=2, markersize=10, label='R² test')
ax2.set_ylabel('R² test', color='crimson')

plt.title('Complejidad del modelo vs. R² en test')
plt.tight_layout(); plt.show()

## 14. Conclusiones

### Preguntas para reflexión con estudiantes

1. **¿Qué modelo obtuvo el mejor R² en test? ¿Por qué crees que es así?**
2. **¿Hay algún modelo con evidencia clara de sobreajuste?** Pista: revisa el `Gap_R2` y las curvas de aprendizaje.
3. **¿La red más compleja gana siempre?** Compara Keras/PyTorch con Random Forest en la tabla de complejidad.
4. **¿Cómo cambiarían los resultados sin winsorización de outliers?** (Experimenta y compara).
5. **¿Cuál es el trade-off entre tiempo de entrenamiento y desempeño?**

### 📚 Notas didácticas
- **MSE** penaliza más fuertemente los errores grandes → sensible a outliers.
- **MAE (MAD)** es más robusto y su valor está en las unidades originales del target.
- **R²** compara con predecir simplemente la media (valor 0). Negativo = peor que la media.
- El gap **train − test** en cualquier métrica es el mejor indicador de sobreajuste.
